In [ ]:
from datetime import datetime
import json
import geopandas as gpd
import pandas as pd
import pyproj
import requests
import shapely.geometry as sg
import shapely
from typing import Dict, Optional, Tuple


# API options
BBOX_REGION = [122300.00,486628.78,122400.00,486475.48]
MISSION_YEAR = 2025 # API tag
SEARCH_RADIUS = 10 # Building search radius from an initial panoramic image location
API_TIMEOUT = 10

WGS84_EPSG = 4326
RD_EPSG = 28992

wgs84_to_rd = pyproj.Transformer.from_crs(WGS84_EPSG, RD_EPSG, always_xy=True).transform
rd_to_wgs84 = pyproj.Transformer.from_crs(RD_EPSG, WGS84_EPSG, always_xy=True).transform


def get_buildings_in_area(target_area: shapely.Geometry) -> gpd.GeoDataFrame:
    """
    Query the BAG api for all buildings that intersect with the given target shape. Note: target shape is expected to be in Amersfoort / RD New format.
    """
    bag_gdf = get_bag_buildings_in_area(target_area=target_area)
    bag3d_df = get_3dbag_buildings_in_area(target_area=target_area)
    return bag_gdf.join(
        other=bag3d_df.set_index("bag_id"),
        on="bag_id",
        how="left",
    )

def get_height_for_id(bag_id: str) -> Tuple[float]:
    "Query the 3DBAG api for the given ID and return ground elevation and building height in meters."

    full_bag_id_str = f"NL.IMBAG.Pand.{bag_id}"
    url = f"https://api.3dbag.nl/collections/pand/items/{full_bag_id_str}"

    with requests.get(url, timeout=API_TIMEOUT) as response:
        json_data = json.loads(response.content)
    
    ground_elevation = json_data["feature"]["CityObjects"][full_bag_id_str]["attributes"]["b3_h_maaiveld"]
    building_height = json_data["feature"]["CityObjects"][full_bag_id_str]["attributes"]["b3_h_dak_max"]
    return (ground_elevation, building_height)

def get_bag_buildings_in_area(target_area: shapely.Geometry) -> gpd.GeoDataFrame:
    date_str = datetime.today().strftime('%Y-%m-%d')
    target_area_wgs84 = shapely.transform(target_area, rd_to_wgs84, interleaved=False)
    url = f"https://api.data.amsterdam.nl/v1/bag/v1/panden?geometrie%5Bintersects%5D={target_area_wgs84.wkt}&geldigOp={date_str}"

    dfs = []

    with requests.get(url, timeout=API_TIMEOUT) as response:
        json_data = json.loads(response.content)
    dfs.append(parse_bag_response(json_data=json_data))
    next_page_url = get_next_page_ams(links=json_data["_links"])

    while next_page_url is not None:
        with requests.get(next_page_url) as response:
            json_data = json.loads(response.content)
        dfs.append(parse_bag_response(json_data=json_data))
        next_page_url = get_next_page_ams(links=json_data["_links"])

    return pd.concat(dfs, ignore_index=True)

def parse_bag_response(json_data: Dict) -> gpd.GeoDataFrame:
    data = {
        "bag_id": [],
        "geometry": [],
    }

    for pand in json_data["_embedded"]["panden"]:
        data["bag_id"].append(pand["identificatie"])
        data["geometry"].append(shapely.from_geojson(json.dumps(pand["geometrie"])))

    return gpd.GeoDataFrame(data=data, crs=RD_EPSG)


def get_3dbag_buildings_in_area(target_area: shapely.Geometry) -> pd.DataFrame:
    minx, miny, maxx, maxy = target_area.bounds
    url = f"https://api.3dbag.nl/collections/pand/items?bbox={minx},{miny},{maxx},{maxy}&limit=100"

    dfs = []

    with requests.get(url, timeout=API_TIMEOUT) as response:
        json_data = json.loads(response.content)
    dfs.append(parse_3dbag_response(json_data=json_data))
    next_page_url = get_next_page_3dbag(links=json_data["links"])

    while next_page_url is not None:
        with requests.get(next_page_url, timeout=API_TIMEOUT) as response:
            json_data = json.loads(response.content)
        dfs.append(parse_3dbag_response(json_data=json_data))
        next_page_url = get_next_page_3dbag(links=json_data["links"])

    return pd.concat(dfs, ignore_index=True)

def parse_3dbag_response(json_data: Dict) -> pd.DataFrame:
    data = {
        "bag_id": [],
        "ground_elevation": [],
        "building_height": [],
    }
    for pand in json_data["features"]:
        id = pand["id"]
        data["bag_id"].append(id.strip("NL.IMBAG.Pand."))
        data["ground_elevation"].append(
            pand["CityObjects"][id]["attributes"]["b3_h_maaiveld"]
        )
        data["building_height"].append(
            pand["CityObjects"][id]["attributes"]["b3_h_dak_max"]
        )
    return pd.DataFrame(data=data)

def get_next_page_3dbag(links: Dict[str, str]) -> Optional[str]:
    for link in links:
        if link["rel"] == "next":
            return link["href"]
    return None

def get_next_page_ams(links: Dict[str, str]) -> Optional[str]:
    if "next" in links.keys():
        return links["next"]["href"]
    else:
        return None

def get_panos_in_area(target_area: shapely.Geometry, mission_year: int) -> gpd.GeoDataFrame:
    """
    Get panoramic image id's for a user defined bounding box region
    """
    minx, miny, maxx, maxy = target_area.bounds
    url = f"https://api.data.amsterdam.nl/panorama/panoramas/?tags=mission-{mission_year}&bbox={minx},{miny},{maxx},{maxy}&srid=28992"

    dfs = []

    with requests.get(url, timeout=API_TIMEOUT) as response:
        json_data = json.loads(response.content)
    dfs.append(parse_panorama_response(json_data=json_data))
    next_page_url = get_next_page_ams(links=json_data["_links"])

    while next_page_url is not None:
        with requests.get(next_page_url) as response:
            json_data = json.loads(response.content)
        dfs.append(parse_panorama_response(json_data=json_data))
        next_page_url = get_next_page_ams(links=json_data["_links"])

    return pd.concat(dfs, ignore_index=True)

def parse_panorama_response(json_data: Dict) -> gpd.GeoDataFrame:
    data = {
        "pano_id": [],
        "full_res_url": [],
        "timestamp": [],
        "geometry": [],
    }
    for pano in json_data["_embedded"]["panoramas"]:
        data["pano_id"].append(pano["pano_id"])
        data["full_res_url"].append(pano["_links"]["equirectangular_full"]["href"])
        data["timestamp"].append(pd.to_datetime(pano["timestamp"]))
        data["geometry"].append(shapely.from_geojson(json.dumps(pano["geometry"])))

    return gpd.GeoDataFrame(data=data, crs=WGS84_EPSG).to_crs(RD_EPSG)

In [ ]:
minx, maxy, maxx, miny = BBOX_REGION
bbox_rd = sg.box(minx, miny, maxx, maxy)

In [ ]:
buildings_gdf = get_buildings_in_area(target_area=bbox_rd)

In [ ]:
panos_gdf = get_panos_in_area(target_area=bbox_rd, mission_year=MISSION_YEAR)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10,8))

buildings_gdf.plot(ax=ax, column="building_height", legend=True)
panos_gdf.plot(ax=ax, legend=False)

plt.show()